In [1]:
import pandas as pd

df = pd.read_csv(
    "exit_annotated_pass1_merged.csv",
      # just read the header, don't load 144k rows
)
print(df.columns.tolist())
print(df["exit_intention_class"].describe())
print(df["exit_level"].value_counts().sort_index())

['row_type', 'post_id', 'comment_id', 'text', 'triage', 'na_subtype', 'triage_reason', 'EX', 'EMO', 'COG', 'MD', 'bat_score', 'EX_reasoning', 'EMO_reasoning', 'COG_reasoning', 'MD_reasoning', 'exit_intention_class', 'exit_confidence', 'exit_evidence_span', 'exit_reasoning', 'exit_level', 'exit_reason_primary', 'exit_reason_secondary']
count      144652
unique          3
top       no_exit
freq       137779
Name: exit_intention_class, dtype: object
Series([], Name: count, dtype: int64)


In [3]:
# -*- coding: utf-8 -*-
"""
check_exit_distribution.py — quick distribution check for exit_intention_class.

Usage:
  python3 check_exit_distribution.py exit_annotated_pass1_merged.csv
"""

import sys
import pandas as pd

def main():
    path =  "exit_annotated_pass1_merged.csv"
    print(f"[i] Loading {path} ...")
    df = pd.read_csv(path, dtype=str)
    print(f"[i] Total rows: {len(df)}\n")

    if "exit_intention_class" not in df.columns:
        print("[X] Column 'exit_intention_class' not found in this file.")
        sys.exit(1)

    counts = df["exit_intention_class"].value_counts(dropna=False)
    pct = df["exit_intention_class"].value_counts(normalize=True, dropna=False) * 100

    print("── exit_intention_class distribution ──────────────")
    for label in counts.index:
        print(f"  {str(label):<25} {counts[label]:>7}   ({pct[label]:.2f}%)")

    n_missing = df["exit_intention_class"].isna().sum()
    if n_missing:
        print(f"\n[!] {n_missing} rows have a missing exit_intention_class.")

    # Optional: also break down by BAT score, if present, since exit intention
    # is expected to correlate with burnout severity
    if "bat_score" in df.columns:
        print("\n── exit_intention_class by bat_score ───────────────")
        df["bat_score_num"] = pd.to_numeric(df["bat_score"], errors="coerce")
        cross = pd.crosstab(df["bat_score_num"], df["exit_intention_class"])
        print(cross)

if __name__ == "__main__":
    main()

[i] Loading exit_annotated_pass1_merged.csv ...
[i] Total rows: 144652

── exit_intention_class distribution ──────────────
  no_exit                    137779   (95.25%)
  exit_contemplating           5354   (3.70%)
  exit_explicit                1519   (1.05%)

── exit_intention_class by bat_score ───────────────
exit_intention_class  exit_contemplating  exit_explicit  no_exit
bat_score_num                                                   
0                                   3009            869   128537
1                                    918            242     6072
2                                    703            166     2174
3                                    565            201      807
4                                    159             41      189


### making the sample for exit human annotation sample


In [1]:
import pandas as pd

# Load the file
df = pd.read_csv("exit_annotated_pass1_merged.csv")

# 1. Check the actual unique values first — run this and confirm the labels match
print(df['exit_intention_class'].value_counts(dropna=False))

exit_intention_class
no_exit               137779
exit_contemplating      5354
exit_explicit           1519
Name: count, dtype: int64


In [3]:
print(df['exit_intention_class'].value_counts(dropna=False))

exit_intention_class
no_exit               137779
exit_contemplating      5354
exit_explicit           1519
Name: count, dtype: int64


In [4]:
import pandas as pd

df = pd.read_csv("exit_annotated_pass1_merged.csv")

# Map your target label -> desired sample size
# EDIT the keys on the left to match exactly what you saw in value_counts() above
sample_plan = {
    "no_exit": 40,
    "exit_contemplating": 30,
    "exit_explicit": 30,
}

samples = []
for label, n in sample_plan.items():
    subset = df[df['exit_intention_class'] == label]
    available = len(subset)
    if available < n:
        print(f" Warning: only {available} rows available for '{label}', requested {n}. Taking all available.")
        n = available
    sampled = subset.sample(n=n, random_state=42)  # fixed seed for reproducibility
    samples.append(sampled)

result = pd.concat(samples, ignore_index=True)

# Shuffle the combined sample so labels aren't grouped in blocks (optional)
result = result.sample(frac=1, random_state=42).reset_index(drop=True)

print(f"Total rows sampled: {len(result)}")
print(result['exit_intention_class'].value_counts())

result.to_csv("exit_sample_100.csv", index=False)
print("Saved to exit_sample_100.csv")

Total rows sampled: 100
exit_intention_class
no_exit               40
exit_explicit         30
exit_contemplating    30
Name: count, dtype: int64
Saved to exit_sample_100.csv
